# Quick Start

This page shows the **traditional first run** for CellPainting-Claw.

The path here is deliberately classical:

- stage or inspect inputs
- work with CellProfiler measurement tables
- build a single-cell table
- run pycytominer aggregation, annotation, normalization, and feature selection
- summarize the resulting classical profiles

DeepProfiler is not part of this page.

All output cells below are real recorded outputs from the current repository demo assets and current runtime.


## Install

From the repository root:


In [ ]:
conda env create -f environment/cellpainting-claw.environment.yml
conda activate cellpainting-claw
pip install -e .[data-access]


## Run Variables


In [ ]:
CONFIG=configs/project_config.demo.json
DATA_ROOT=demo/workspace/outputs/quick_start_data
RUN_ROOT=demo/workspace/outputs/quick_start_classical


## Input Staging

If your inputs still live in the Cell Painting Gallery, the normal entry sequence is:

1. inspect configured sources
2. plan a download
3. download a bounded slice into local storage

The commands below were run against the live Cell Painting Gallery with the demo config.


### Inspect Configured Sources


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill data-inspect-availability \
  --output-dir "$DATA_ROOT/01_inspect"


{
  "dataset_id": "cpg0016-jump",
  "gallery_dataset_count": 42,
  "gallery_source_count": 14,
  "first_sources": [
    "source_1",
    "source_10",
    "source_11",
    "source_13",
    "source_15"
  ],
  "required_packages_ok": true
}


### Plan A Download


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill data-plan-download \
  --dataset-id cpg0016-jump \
  --source-id source_4 \
  --max-files 4 \
  --output-dir "$DATA_ROOT/02_plan"


{
  "resolved_dataset_id": "cpg0016-jump",
  "resolved_source_id": "source_4",
  "resolved_prefix": "cpg0016-jump/source_4/",
  "step_count": 1,
  "max_files": 4
}


### Download A Small Local Input Slice


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill data-download \
  --dataset-id cpg0016-jump \
  --source-id source_4 \
  --subprefix workspace/load_data_csv/2021_04_26_Batch1/BR00117035 \
  --output-dir "$DATA_ROOT/03_download_small"


{
  "prefix": "cpg0016-jump/source_4/workspace/load_data_csv/2021_04_26_Batch1/BR00117035/",
  "matched_object_count": 2,
  "downloaded_count": 2,
  "object_keys": [
    ".../load_data.csv",
    ".../load_data_with_illum.csv"
  ]
}


This bounded download is only an input-staging example. The traditional profiling demo below uses the repository demo backend, which already includes a minimal local CellProfiler result set.


## Measurement Tables

The first classical step is to expose the CellProfiler measurement tables.


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill cp-extract-measurements \
  --output-dir "$RUN_ROOT/01_measurements"


{
  "mode": "bundled-demo-outputs",
  "image_table_path": ".../Image.csv",
  "cells_table_path": ".../Cells.csv",
  "nuclei_table_path": ".../Nuclei.csv"
}


In this public demo checkout, the profiling backend script itself is not packaged. The skill therefore reuses the bundled demo measurement tables instead of rerunning CellProfiler. For user-owned workspaces, this same skill is still the public entrypoint for the measurement stage.


## Single-Cell Table


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill cp-build-single-cell-table \
  --image-csv-path demo/backend/profiling_backend/outputs/cellprofiler/Image.csv \
  --object-table-path demo/backend/profiling_backend/outputs/cellprofiler/Cells.csv \
  --object-table Cells \
  --output-dir "$RUN_ROOT/02_single_cell"


{
  "row_count": 4,
  "column_count": 16,
  "object_table": "Cells"
}


This produces one merged single-cell table that pycytominer can consume directly.


## Classical Profiles


### Aggregate Profiles


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill cyto-aggregate-profiles \
  --single-cell-path "$RUN_ROOT/02_single_cell/single_cell.csv.gz" \
  --output-dir "$RUN_ROOT/03_cyto_aggregate"


{
  "row_count": 2,
  "column_count": 14
}


### Annotate Profiles


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill cyto-annotate-profiles \
  --aggregated-path "$RUN_ROOT/03_cyto_aggregate/pycytominer/aggregated.parquet" \
  --output-dir "$RUN_ROOT/04_cyto_annotate"


{
  "row_count": 2,
  "column_count": 17
}


### Normalize Profiles


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill cyto-normalize-profiles \
  --annotated-path "$RUN_ROOT/04_cyto_annotate/pycytominer/annotated.parquet" \
  --output-dir "$RUN_ROOT/05_cyto_normalize"


{
  "row_count": 2,
  "column_count": 17
}


### Select Profile Features


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill cyto-select-profile-features \
  --normalized-path "$RUN_ROOT/05_cyto_normalize/pycytominer/normalized.parquet" \
  --output-dir "$RUN_ROOT/06_cyto_select"


{
  "row_count": 2,
  "column_count": 12
}


These four pycytominer stages take the run from raw single-cell measurements to a smaller feature-selected well-level profile table.


## Summary Outputs


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill cyto-summarize-classical-profiles \
  --feature-selected-path "$RUN_ROOT/06_cyto_select/pycytominer/feature_selected.parquet" \
  --output-dir "$RUN_ROOT/07_cyto_summary"


{
  "row_count": 2,
  "feature_count": 6,
  "top_feature_count": 6,
  "pca_components": 2
}


This final step writes the first human-readable review layer for the classical profile branch.


## Result Files

After this Quick Start, the most useful files to inspect are:

- `data_access_summary.json` for the configured source inventory
- `download_plan.json` for the resolved Gallery request
- `single_cell.csv.gz` for the merged single-cell measurements table
- `aggregated.parquet`, `annotated.parquet`, `normalized.parquet`, and `feature_selected.parquet` for the pycytominer stages
- `profile_summary.json`, `well_metadata_summary.csv`, `top_variable_features.csv`, and `pca_plot.png` for the final review layer


## Next Pages

Continue with these pages after the first run:

- [CellPainting-Skills](../skills/index.md) for the full skill catalog
- [Command-Line Interface](../cli/index.md) for direct CLI usage
- [OpenClaw](../openclaw/index.md) for agent-mediated use of the same skills
